In [ ]:
import napari
import tifffile
import numpy as np
import pandas as pd
import scipy as sc
import re
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from cellpose.io import imread
from skimage.io.collection import alphanumeric_key
from skimage.registration import phase_cross_correlation
from copy import copy
from dask import delayed
import dask.array as da
from glob import glob
from napari_animation import Animation
import nd2_dask 
import os
from scipy import signal, ndimage, optimize
from skimage.filters import threshold_otsu, gaussian, threshold_yen, threshold_triangle, threshold_mean
from skimage.segmentation import watershed, clear_border
from skimage.feature import peak_local_max
from skimage import color, morphology
from skimage.measure import label, regionprops, find_contours
from skimage.color import label2rgb
import seaborn as sns
from sklearn.cluster import KMeans
from skimage.filters import try_all_threshold
import RedLionfishDeconv as rl
from microfilm import microplot, colorify
from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
import matplotlib as mpl

In [ ]:
import numpy as np
from skimage.filters import threshold_otsu, threshold_mean, threshold_yen, threshold_triangle
from skimage import morphology
from skimage.measure import label, regionprops
from skimage.segmentation import clear_border
from scipy import optimize

def extract_actin_cloud(image, threshFactor=1, smallObjectSize=50, removeSmall=True, onlyBiggest=False):
    """
    Extract actin structures using Otsu thresholding and morphological operations.

    Parameters:
        image (ndarray): Input grayscale image.
        threshFactor (float): Multiplier for the Otsu threshold. Default is 1.
        smallObjectSize (int): Minimum size of objects to keep. Default is 50.
        removeSmall (bool): Whether to remove small objects. Default is True.
        onlyBiggest (bool): If True, keeps only the largest connected component.

    Returns:
        ndarray: Labeled image of segmented actin structures.
    """
    thresh = threshold_otsu(image)
    binary = image > (thresh * threshFactor)
    cleaned = morphology.remove_small_objects(binary, min_size=smallObjectSize) if removeSmall else binary
    cleaned = morphology.binary_opening(cleaned) if removeSmall else cleaned
    cleaned = clear_border(cleaned)
    label_image = label(cleaned)
    if onlyBiggest:
        label_image = (label_image == np.bincount(label_image.ravel())[1:].argmax() + 1).astype(int)
    return label_image

def extract_actin_cloud_new(image, threshFactor=1, smallObjectSize=50, removeSmall=True, onlyBiggest=False):
    """
    Variant of actin cloud extraction without border clearing.

    Parameters and Returns are same as extract_actin_cloud.
    """
    thresh = threshold_otsu(image)
    binary = image > (thresh * threshFactor)
    cleaned = morphology.remove_small_objects(binary, min_size=smallObjectSize) if removeSmall else binary
    cleaned = morphology.binary_opening(cleaned) if removeSmall else cleaned
    label_image = label(cleaned)
    if onlyBiggest:
        label_image = (label_image == np.bincount(label_image.ravel())[1:].argmax() + 1).astype(int)
    return label_image

def extract_cell(image, threshFactor=1, threshold_mode="otsu", thresholdInput=None,
                 smallObjectSize=50, removeSmall=True, onlyBiggest=False, closingSize=50):
    """
    Segment cells using selectable thresholding and morphological filters.

    Parameters:
        image (ndarray): Input grayscale image.
        threshFactor (float): Multiplier for threshold value.
        threshold_mode (str): Thresholding method ('otsu', 'mean', 'yen', 'triangle', 'input').
        thresholdInput (float): Manual threshold if threshold_mode == 'input'.
        smallObjectSize (int): Minimum size of objects to retain.
        removeSmall (bool): Whether to remove small objects.
        onlyBiggest (bool): Whether to keep only the largest object.
        closingSize (int): Size of closing operation (not used by default).

    Returns:
        ndarray: Labeled image of segmented cells.
    """
    threshold_map = {
        "otsu": threshold_otsu,
        "mean": threshold_mean,
        "yen": threshold_yen,
        "triangle": threshold_triangle,
    }
    thresh = thresholdInput if threshold_mode == "input" else threshold_map[threshold_mode](image)
    binary = image > (thresh * threshFactor)
    cleaned = morphology.remove_small_objects(binary, min_size=smallObjectSize) if removeSmall else binary
    label_image = label(cleaned)
    if onlyBiggest:
        label_image = (label_image == np.bincount(label_image.ravel())[1:].argmax() + 1).astype(int)
    return label_image

def extract_nucleus(image, threshFactor=1, smallObjectSize=50, removeSmall=True, closingSize=4):
    """
    Segment nuclei from an image using thresholding and morphological operations.

    Parameters:
        image (ndarray): Input grayscale image.
        threshFactor (float): Multiplier for the Otsu threshold.
        smallObjectSize (int): Minimum object size to retain.
        removeSmall (bool): Remove small objects if True.
        closingSize (int): Disk size for morphological closing.

    Returns:
        ndarray: Labeled binary image of nuclei.
    """
    thresh = threshold_otsu(image)
    binary = image > (thresh * threshFactor)
    cleaned = morphology.remove_small_objects(binary, min_size=smallObjectSize) if removeSmall else binary
    cleaned = morphology.binary_opening(cleaned) if removeSmall else cleaned
    cleaned = morphology.closing(cleaned, morphology.disk(closingSize))
    return label(cleaned)

def filter_objects_by_size(label_image, min_size=0, max_size=None):
    """
    Filters labeled objects by size range.

    Parameters:
        label_image (ndarray): Input labeled image.
        min_size (int): Minimum object size to keep.
        max_size (int or None): Maximum object size to keep (None = no max).

    Returns:
        ndarray: Filtered labeled image.
    """
    small_removed = morphology.remove_small_objects(label_image, min_size)
    if max_size is not None:
        mid_removed = morphology.remove_small_objects(small_removed, max_size)
        return small_removed - mid_removed
    return small_removed

def cross_correlation(x, y, maxLag=None, lagStep=1):
    """
    Compute cross-correlation between two 1D signals.

    Parameters:
        x (array_like): First signal.
        y (array_like): Second signal.
        maxLag (int): Maximum lag to evaluate.
        lagStep (int): Step size between lags.

    Returns:
        list: Correlation values for each lag.
    """
    if maxLag is None:
        maxLag = len(x) // 4
    x = np.array(x)
    y = np.array(y)
    if len(y) < maxLag + 1:
        raise ValueError("y array is too short.")
    mx, my = np.mean(x), np.mean(y)
    denominator = np.sqrt(np.sum((x - mx)**2) * np.sum((y - my)**2))
    xcorr = []
    for i in range(0, maxLag + 1, lagStep):
        nom = np.sum((x[:len(x) - i] - mx) * (y[i:] - my))
        xcorr.append(nom / denominator)
    return xcorr

def gaus(x, a, x0, sigma, b):
    """
    Gaussian function with offset.

    Parameters:
        x (array): Independent variable.
        a (float): Amplitude.
        x0 (float): Center.
        sigma (float): Standard deviation.
        b (float): Baseline.

    Returns:
        array: Gaussian evaluated at x.
    """
    return a * np.exp(-(x - x0)**2 / (2 * sigma**2)) + b

def cross_correlation_symmetric(x, y):
    """
    Compute symmetric cross-correlation with negative and positive lags.

    Parameters:
        x, y (array_like): Input signals.

    Returns:
        tuple: (lags, correlation values)
    """
    xcorr = cross_correlation(x, y)
    xcorrNeg = cross_correlation(y, x)[::-1]
    lagxcorr = np.arange(0, len(xcorr))
    lagxcorrneg = -np.arange(0, len(xcorrNeg))[::-1]
    return np.concatenate((lagxcorrneg[:-1], lagxcorr)), np.concatenate((xcorrNeg[:-1], xcorr))

def bead_center_plane_gaussian(beadImage, timeFrame):
    """
    Fit bead side projection to a Gaussian to find center.

    Parameters:
        beadImage (ndarray): 4D bead image (T, Z, Y, X).
        timeFrame (int): Time index.

    Returns:
        float: Estimated center of bead.
    """
    beadSide = np.mean(beadImage[timeFrame], axis=(2,1))
    x = np.arange(len(beadSide))
    try:
        fit = optimize.curve_fit(gaus, x, beadSide, p0=[100, 15, 5, 200])[0]
    except:
        fit = (0, 0, 0)
    return fit[1]

def bead_center_plane_max_intensity(beadImage, timeFrame):
    """
    Return the Z-plane with max total intensity.

    Parameters:
        beadImage (ndarray): 4D bead image.
        timeFrame (int): Time index.

    Returns:
        int: Z-index of max intensity plane.
    """
    maxPlane = np.argmax([np.sum(beadImage[timeFrame, z]) for z in range(beadImage.shape[1])])
    return maxPlane

def bead_center_plane_correlation(beadImage, timeFrame1, timeFrame2):
    """
    Estimate displacement between bead profiles via correlation.

    Parameters:
        beadImage (ndarray): 4D bead image.
        timeFrame1, timeFrame2 (int): Time indices.

    Returns:
        float: Estimated shift (from fitted parabola).
    """
    b1 = np.mean(beadImage[timeFrame1], axis=(2,1))
    b2 = np.mean(beadImage[timeFrame2], axis=(2,1))
    lags, corr = cross_correlation_symmetric(b1, b2)
    try:
        fit = optimize.curve_fit(lambda x, a, b, x0: a + b*(x - x0)**2, lags, corr, p0=(0.8, -0.01, 0.2))[0]
    except:
        fit = (0, 0, 0)
    return fit[2]

def bead_center_plane_segment_centroid(beadImage, timeFrame):
    """
    Segment bead side view and compute weighted centroid.

    Parameters:
        beadImage (ndarray): 4D bead image.
        timeFrame (int): Time index.

    Returns:
        float: X-centroid of bead.
    """
    beadSide = np.mean(beadImage[timeFrame], axis=2)
    thresh = threshold_otsu(beadSide)
    bw = morphology.closing(beadSide > thresh, morphology.disk(4))
    label_image = label(bw)
    return regionprops(label_image, beadSide)[0].centroid_weighted[0]

def calculate_cropped_coordinates_from_mask(imageMask, label_id, margin):
    """
    Get cropped bounding box slice from labeled image.

    Parameters:
        imageMask (ndarray): Labeled image.
        label_id (int): Label index to crop.
        margin (int): Padding around object.

    Returns:
        tuple: Slices for cropping.
    """
    r, c = map(int, regionprops(imageMask, imageMask)[label_id].centroid)
    return (slice(max(r - margin // 2, 0), r + margin // 2),
            slice(max(c - margin // 2, 0), c + margin // 2))

def extract_single_bead_XY(singleBeadImage):
    """
    Get centroid of single bead from binary segmentation.

    Parameters:
        singleBeadImage (ndarray): 2D image.

    Returns:
        tuple: (Y, X) centroid coordinates.
    """
    binary = singleBeadImage > threshold_otsu(singleBeadImage)
    labelBead = label(binary)
    return regionprops(labelBead, singleBeadImage)[0].centroid_weighted

def calculate_nucleus_mask(imageData, timeStart):
    """
    Generate binary nucleus masks across time.

    Parameters:
        imageData (ndarray): 5D image (T, Z, Y, X, C).
        timeStart (int): Time index to start segmentation.

    Returns:
        list: List of 2D binary masks.
    """
    masks = []
    for t in range(imageData.shape[0]):
        if t


In [ ]:
def analyze_bead_displacement_center_boxes(
    imageData, beadsLabeledMask, marginBead=5, margins=[5, 20, 50],
    calculateNucleusMask=False, calculateActinMask=False, nucleusTimeStart=0,
    beadPlaneModes=["Centroid", "MaxIntensity"]
):
    """
    Analyze bead displacement using different center-plane detection methods
    and compute associated intensity profiles.

    Parameters:
        imageData (ndarray): 5D array of imaging data [time, z, y, x, channels].
        beadsLabeledMask (ndarray): Labeled mask of bead locations.
        marginBead (int): Margin for bead cropping.
        margins (list): List of margins for intensity box calculations.
        calculateNucleusMask (bool): If True, compute and analyze nucleus mask.
        calculateActinMask (bool): If True, compute and analyze actin mask.
        nucleusTimeStart (int): Time index to start nucleus mask calculation.
        beadPlaneModes (list): Methods to estimate bead center plane.

    Returns:
        data (DataFrame): Compiled measurements of beads over time.
        newLabels (ndarray): Updated labeled mask with contiguous IDs.
    """

    
    data = pd.DataFrame({"beadID": [], "beadCoords":[], "beadInPath":[], "beadR":[], "beadC":[], "distanceCellCenter":[],
                        })
    for margin in margins:
        data["actinIntensity_"+str(margin)+"_box"] = []
        data["nucleusIntensity_"+str(margin)+"_box"] = []
    
    modeMappingFunctions = {"Centroid": bead_center_plane_segment_centroid,
                           "MaxIntensity": bead_center_plane_max_intensity,
                           "GaussFit": bead_center_plane_gaussian,
                           "Correlation": lambda beadImage, timeFrame: bead_center_plane_correlation(beadImage, timeFrame1=0, timeFrame2=timeFrame)}
    
    for mode in beadPlaneModes:
        data["beadPlane"+mode] = []
            
    newLabels = copy(beadsLabeled)
    #newLabels[newLabels==0] = -1
                        
    if calculateNucleusMask:
        data["nucleusMaskOverlap"] = []
        nucleusMasks = calculate_nucleus_mask(imageData, nucleusTimeStart)
    if calculateActinMask:
        data["actinMaskOverlap"] = []
        actinMasks = calculate_actin_mask(imageData, 0)
    
    
    
    
    #Get the distance from the cell center
    distanceMaps = []

    for time in np.arange(0,imageData.shape[0]):   
        labelImage = extract_cell(np.mean(imageData[time, :15,:,:, 1], axis=0), threshFactor=0.97, threshold_mode="otsu")
        labelImage = ndimage.binary_fill_holes(labelImage).astype(int)
        distance = ndimage.distance_transform_edt(labelImage).astype(int)
        distanceMaps.append(distance)                         
        

    
    for lab, beadId in enumerate(np.unique(beadsLabeledMask)[1:]):
        
        nucleusIntensity = {key: [] for key in margins}                
        nucleusMaskIntensity = {key: [] for key in margins}                
        actinMaskIntensity = {key: [] for key in margins}
        actinIntensity = {key: [] for key in margins}
        croppedCoords = {key: [] for key in margins}
        modeMappingPlanes = {mode: [] for mode in beadPlaneModes}
        
        newLabels[newLabels==beadId] = lab + 1
        
        beadRs = []
        beadCs = []
        
        distancesCenterCell = []
        for time in np.arange(0,imageData.shape[0]):
            for margin in margins:
                croppedCoords = calculate_cropped_coordinates_from_mask(beadsLabeledMask, label=lab, margin=margin)
                croppedBeadCoords = calculate_cropped_coordinates_from_mask(beadsLabeledMask, label=lab, margin=marginBead)

                beadImage = imageData[:,:, croppedBeadCoords[0], croppedBeadCoords[1],0]
                nucleusImage = imageData[:,:, croppedCoords[0], croppedCoords[1],2]
        
                actinImage = imageData[:,:, croppedCoords[0], croppedCoords[1],1]
                nucleusIntensity[margin].append(np.sum(nucleusImage[time,:]))
                     
                if calculateNucleusMask:
                    nucleusMaskIntensityMargin.append(np.sum(nucleusMasks[time][croppedCoords[0], croppedCoords[1]]))
                if calculateActinMask:
                    actinMaskIntensityMargin.append(np.sum(actinMasks[time][croppedCoords[0], croppedCoords[1]]))

                actInt = np.sum(actinImage[time,:])
                actinIntensity[margin].append(actInt)
                
            distanceImage = distanceMaps[time][croppedCoords[0], croppedCoords[1]]
            distancesCenterCell.append(np.mean(distanceImage))
            for mode in beadPlaneModes:
                modeMappingPlanes[mode].append(modeMappingFunctions[mode](beadImage, time))

            beadR, beadC = extract_single_bead_XY(np.sum(beadImage[time, :, :, :], axis=0))
            beadRs.append(beadR)
            beadCs.append(beadC)
            
            
        subData = pd.DataFrame({"beadID": imageData.shape[0]*[lab+1], "coords":imageData.shape[0]*[croppedCoords], 
                                "beadCoords":imageData.shape[0]*[croppedBeadCoords], "beadInPath": imageData.shape[0]*[False],
                                "beadR": beadRs,
                                "beadC": beadCs,
                               "distanceCenterCell": distancesCenterCell,
                               })
        for margin in margins:
            subData["actinIntensity_"+str(margin)+"_box"] = actinIntensity[margin]
            subData["nucleusIntensity_"+str(margin)+"_box"] = nucleusIntensity[margin]
                                
        if calculateNucleusMask:
            subData["nucleusMaskOverlap"] = nucleusMaskIntensity
        if calculateActinMask:
            subData["actinMaskOverlap"] = actinMaskIntensity
                
        for mode in beadPlaneModes:
            subData["beadPlane"+mode] = modeMappingPlanes[mode]
        
        data = pd.concat((data, subData))
    data["beadPlaneKymo"] = 0
    for bead in data["beadID"].unique():

        images = imageData[:,:,:,:,0]
        kymo = []
        for time in data.index.unique():
            sliceCoords = data.loc[data["beadID"]==bead, "beadCoords"][time]
            kymo.append(np.mean(images[time,:,sliceCoords[0], sliceCoords[1]], axis=(1,2)))
        kymo=np.transpose(np.asarray(kymo))
        #plt.plot(np.mean(images[0,:,sliceCoords[0], sliceCoords[1], 0], axis=(1,2)))


        binaryKymo = (kymo>threshold_otsu(kymo)).astype(int)
        #binaryKymo = kymo>np.mean(kymo)


        #print(np.argmin(np.mean(binaryKymo, axis=1)))
        #if np.argmin(np.mean(binaryKymo, axis=1))< 20:
        if np.unravel_index(kymo.argmin(), kymo.shape)[0] < 20:
            pass
            edge = np.argmax(binaryKymo, axis=0)
        else:
            #binaryKymo = np.invert(binaryKymo)
            binaryKymo = np.flip(binaryKymo, axis=0)
            edge = binaryKymo.shape[0] - np.argmax(binaryKymo, axis=0)
            binaryKymo = np.flip(binaryKymo, axis=0)
        data.loc[data["beadID"]==bead, "beadPlaneKymo"] = edge
    return data, newLabels

In [ ]:
def plot_bead_box(
    data, imageData, beadMaskImage, bead, time, margin=50,
    plotMode="actin", nucleusMask=False, actinMask=False, nucleusTimeStart=0
):
    """
    Visualize a box around a bead with optional overlays for actin/nucleus masks.

    Parameters:
        data (DataFrame): Bead analysis results.
        imageData (ndarray): Imaging data [time, z, y, x, channels].
        beadMaskImage (ndarray): Mask showing labeled beads.
        bead (int): Bead ID to plot.
        time (int): Time index.
        margin (int): Additional margin around the bead box.
        plotMode (str): "actin" or "nucleus" to determine overlay image.
        nucleusMask (bool): Overlay nucleus mask if True.
        actinMask (bool): Overlay actin mask if True.
        nucleusTimeStart (int): Start time for mask calculation.

    Returns:
        None (Displays a matplotlib figure).
    """
    
    fig = plt.figure(figsize=(7,7))
    ax = fig.add_subplot(111)
    singleBead = copy(beadMaskImage)
    singleBead[singleBead != bead] = 0
    ax.imshow(singleBead)
    rCoordsOld = data.loc[data["beadID"] == bead, "coords"][0][0]
    cCoordsOld = data.loc[data["beadID"] == bead, "coords"][0][1]
    rCoords = slice(np.max([0, rCoordsOld.start - margin//2]), np.min([imageData.shape[2], rCoordsOld.stop + margin//2]))
    cCoords = slice(np.max([0, cCoordsOld.start - margin//2]), np.min([imageData.shape[3], cCoordsOld.stop + margin//2]))
    
    imageContour = singleBead[rCoords, cCoords]
    
    contour = np.transpose(find_contours(imageContour)[0])
    ax.plot(contour[1], contour[0], color="yellow", lw=1.5)
    #contour = np.transpose(find_contours(images[0,0, data.loc[data["beadID"] == bead, "coords"][0][0], data.loc[data["beadID"] == bead, "coords"][0][1],0]))
    #imageContour = np.mean(images[time,:, data.loc[data["beadID"] == bead, "coords"][0][0], 
    #                              data.loc[data["beadID"] == bead, "coords"][0][1],0], axis=0)
    #contour = np.transpose(find_contours(imageContour))
    
    #plt.imshow(np.mean(images[time,:, data.loc[data["beadID"] == bead, "coords"][0][0], data.loc[data["beadID"] == bead, "coords"][0][1],2], axis=0))
    if plotMode == "actin":
        ax.imshow(np.mean(images[time,:, rCoords, cCoords, 1], axis=0), cmap="viridis")
        if actinMask:
            actinMasks = calculate_actin_mask(imageData, nucleusTimeStart)
            actinMaskImage = actinMasks[time][rCoords, cCoords]
            try:

                contourActin = np.transpose(find_contours(actinMaskImage)[0])
                ax.plot(contourActin[1], contourActin[0], color="green", lw=1.5)
            except:
                pass
            ax.imshow(actinMaskImage, alpha=0.5, cmap="viridis")
        
    elif plotMode == "nucleus":
        if nucleusMask:
            try:
                nucleusMasks = calculate_nucleus_mask(imageData, nucleusTimeStart)
                nucleusMaskImage = nucleusMasks[time][rCoords, cCoords]
                contourNucleus = np.transpose(find_contours(nucleusMaskImage)[0])
                ax.plot(contourNucleus[1], contourNucleus[0], color="green", lw=1.5)
            except:
                pass
            ax.imshow(nucleusMaskImage, alpha=0.8, cmap="Blues")
        ax.imshow(np.mean(images[time,:, rCoords, cCoords, 2], axis=0), cmap="Blues")
    beadsMask = copy(beadsLabeled[rCoords, cCoords])
    alphas = np.zeros(imageContour.shape)
    alphas[beadsMask != 0] = 1
    ax.imshow(alphas, alpha=alphas, cmap="autumn_r")

    rect = Rectangle((np.min([cCoordsOld.start, margin//2]), np.min([rCoordsOld.start, margin//2])),
                     cCoordsOld.stop - cCoordsOld.start, 
                     rCoordsOld.stop - rCoordsOld.start, edgecolor = 'white', fill=False)
    
    ax.add_patch(rect)
    plt.show()

In [ ]:
def plot_profiles(
    data, bead, beadPlane="beadPlaneCentroid", margin=20,
    nucleusIntensity="nucleusMaskOverlap", actinIntensity="actinIntensity",
    savefig=False, savefigName=None
):
    """
    Plot Z-plane height with actin and nucleus intensities for a specific bead.

    Parameters:
        data (DataFrame): Bead analysis results.
        bead (int): ID of the bead to plot.
        beadPlane (str): Column name for Z-plane data.
        margin (int): Margin used for intensity calculations.
        nucleusIntensity (str): Base name for nucleus intensity column.
        actinIntensity (str): Base name for actin intensity column.
        savefig (bool): Whether to save the figure.
        savefigName (str): Path to save the figure.

    Returns:
        None (Displays a matplotlib figure).
    """
    
    actinIntensity +="_"+str(margin)+"_box"
    nucleusIntensity +="_"+str(margin)+"_box"
    fig = plt.figure(figsize=(6,4))
    bm = data["beadID"] == bead
    rangeBeads = data.loc[bm, beadPlane].max() - data.loc[bm, beadPlane].min()
    rangeActin = data.loc[bm, actinIntensity].max() - data.loc[bm, actinIntensity].min()
    rangeNucleus = data.loc[bm, nucleusIntensity].max() - data.loc[bm, nucleusIntensity].min()
    
    if rangeBeads == 0:
        factorActin = 1
    else:
        factorActin = rangeActin/rangeBeads
        factorNucleus = rangeNucleus/rangeBeads

    #plt.plot(data.loc[data["beadID"] == bead, "zPlane"] - np.mean(data.loc[data["beadID"] == bead, "zPlane"]), color="purple")
    #plt.ylabel("Bead height")
    plt.xlabel("Time")
    plt.plot(data.loc[bm, beadPlane] - np.mean(data.loc[bm, beadPlane]), 
             color="#333333", lw=3, label="Bead Z")
    plt.fill_between(np.arange(len(data.loc[bm, beadPlane])),
                    np.min(data.loc[bm, beadPlane] - np.mean(data.loc[bm, beadPlane])),
                           data.loc[bm, beadPlane] - np.mean(data.loc[bm, beadPlane]),
                    color="k", alpha=0.1)
    #plt.plot(data.loc[data["beadID"] == bead, "zPlane3"] - np.mean(data.loc[data["beadID"] == bead, "zPlane3"]))
    plt.plot((data.loc[bm,actinIntensity]-np.mean(data.loc[bm,actinIntensity]))/factorActin, 
             color="#d85252", lw=4, label="Actin")
    plt.plot((data.loc[bm, nucleusIntensity]-np.mean(data.loc[bm, nucleusIntensity]))/factorNucleus,
             "--", lw=4, label="Nucleus")
    plt.legend()
    if savefig:
        plt.savefig(savefigName, dpi=300)
    plt.show()

In [ ]:
def export_analyzed_data(images, data, labeledBeads, folder, fileName, compression=0.2, exportOnlyData=False):
    """
    Export bead analysis data and optionally compressed image stacks.

    Parameters:
        images (ndarray): Imaging data.
        data (DataFrame): Analysis results.
        labeledBeads (ndarray): Mask of labeled beads.
        folder (str): Directory to save outputs.
        fileName (str): Base filename.
        compression (float): Downsampling factor.
        exportOnlyData (bool): If True, skip image export.

    Returns:
        None
    """
    
    data.to_csv(folder+fileName[:-4]+"_data.csv")
    if not exportOnlyData:
        smallImageActin = np.array([ndimage.zoom(np.sum(images[time,:,:,:,1], axis=0), zoom=compression) for time in np.arange(images.shape[0])])
        smallImageNucleus = np.array([ndimage.zoom(np.sum(images[time,:,:,:,2], axis=0), zoom=compression) for time in np.arange(images.shape[0])])
        smallImage = np.moveaxis(np.array([smallImageActin, smallImageNucleus]), 0, -1)
        tifffile.imwrite(folder+fileName[:-4]+"_lowres_compression_"+str(compression)+".tif", smallImage)
        tifffile.imwrite(folder+fileName[:-4]+"_beads_labels.tif", labeledBeads)


        
def import_analyzed_data(folder, fileName):
    """
    Load previously exported bead analysis data and images.

    Parameters:
        folder (str): Directory of saved data.
        fileName (str): Base filename of saved data.

    Returns:
        images (ndarray): Upscaled actin and nucleus images.
        newLabels (ndarray): Loaded labeled mask.
        data (DataFrame): Loaded analysis results.
    """

    if fileName.endswith(".tif"):
        fileName = fileName[:-4]
    compressedFileName = [filename for filename in os.listdir(folder) if filename.startswith(fileName+"_lowres_compression")][0]
    images = imread(folder+compressedFileName)
    
    compression = float(re.findall("\d+\.\d+", compressedFileName[-10:])[0])
    
    
    actinBig = [ndimage.zoom(images[time, :,:,0], zoom=1/compression) for time in np.arange(images.shape[0])]
    nucleusBig = [ndimage.zoom(images[time, :,:,1], zoom=1/compression) for time in np.arange(images.shape[0])]
    newLabels = imread(folder+fileName+"_beads_labels.tif")
    data = pd.read_csv(folder+fileName+"_data.csv", index_col=0)
    return np.moveaxis(np.array([actinBig, nucleusBig]), 0, -1), newLabels, data


def extract_window_around_bead(data):
    """
    Determine the largest window size used in any bead crop.

    Parameters:
        data (DataFrame): Bead analysis results.

    Returns:
        size (int): Maximum window size found.
    """

    size = 0
    for bead_id in data["beadID"].unique():

        coordsSlice = data.loc[data["beadID"]==bead_id, "coords"][0][0]
        print(coordsSlice)
        if (coordsSlice.stop - coordsSlice.start) > size:
            print("yes")
            size = coordsSlice.stop - coordsSlice.start
    return size



def classify_actin_contribution(data, cloud=True, boxMargin=20, chunkTime=None, chunkBead=None):
    """
    Classify actin signal contribution using KMeans clustering.

    Parameters:
        data (DataFrame): Bead analysis results.
        cloud (bool): Whether to use 3 clusters (True) or 2 (False).
        boxMargin (int): Margin used for actin box.
        chunkTime (int or None): Time threshold for temporal split.
        chunkBead (int or None): Bead ID threshold for spatial split.

    Returns:
        Series: Cluster labels indicating bead path classification.
    """


    if cloud:
        clusters = 3
    else:
        clusters = 2
    means = KMeans(clusters)
    channel = "actinIntensity_"+str(boxMargin)+"_box"
    clusterData = pd.DataFrame({"beadID": data["beadID"]}, index=data.index, )
    if not chunkTime and not chunkBead:
        
        cluster_id = means.fit_predict(data[channel].values.reshape(-1,1))
        ord_idx=np.argsort(means.cluster_centers_.flatten())
        cntrs = np.zeros_like(cluster_id)-1
        for i in np.arange(clusters):
            cntrs[cluster_id==ord_idx[i]]=i

        clusterData["beadInPath"] = cntrs
        
    elif chunkTime:
        
        cluster_id = means.fit_predict(data.loc[data.index<=chunkTime, channel].values.reshape(-1,1))
        ord_idx=np.argsort(means.cluster_centers_.flatten())
        cntrs = np.zeros_like(cluster_id)-1
        for i in np.arange(clusters):
            cntrs[cluster_id==ord_idx[i]]=i
        clusterData.loc[clusterData.index<=chunkTime, "beadInPath"] = cntrs
        cluster_id = means.fit_predict(data.loc[data.index>chunkTime, channel].values.reshape(-1,1))
        ord_idx=np.argsort(means.cluster_centers_.flatten())
        cntrs = np.zeros_like(cluster_id)-1
        for i in np.arange(clusters):
            cntrs[cluster_id==ord_idx[i]]=i
        clusterData.loc[clusterData.index>chunkTime, "beadInPath"] = cntrs
    elif chunkBead:
        cluster_id = means.fit_predict(data.loc[data["beadID"]<=chunkBead, channel].values.reshape(-1,1))
        ord_idx=np.argsort(means.cluster_centers_.flatten())
        cntrs = np.zeros_like(cluster_id)-1
        for i in np.arange(clusters):
            cntrs[cluster_id==ord_idx[i]]=i
        clusterData.loc[clusterData["beadID"]<=chunkBead, "beadInPath"] = cntrs
        
        cluster_id = means.fit_predict(data.loc[data["beadID"]>chunkBead, channel].values.reshape(-1,1))
        ord_idx=np.argsort(means.cluster_centers_.flatten())
        cntrs = np.zeros_like(cluster_id)-1
        for i in np.arange(clusters):
            cntrs[cluster_id==ord_idx[i]]=i
        clusterData.loc[clusterData["beadID"]>chunkBead, "beadInPath"] = cntrs
            
        

            
    return clusterData["beadInPath"]


def classify_nucleus_contribution(data, boxMargin=20, clusters=2, chunkTime=None, chunkBead=None):
    """
    Classify nucleus signal contribution using KMeans clustering.

    Parameters:
        data (DataFrame): Bead analysis results.
        boxMargin (int): Margin used for nucleus box.
        clusters (int): Number of KMeans clusters.
        chunkTime (int or None): Time threshold for temporal split.
        chunkBead (int or None): Bead ID threshold for spatial split.

    Returns:
        Series: Cluster labels indicating nuclear overlap classification.
    """

    
    means = KMeans(clusters)
    channel = "nucleusIntensity_"+str(boxMargin)+"_box"
    clusterData = pd.DataFrame({"beadID": data["beadID"]}, index=data.index, )
    if not chunkTime and not chunkBead:
        
        cluster_id = means.fit_predict(data[channel].values.reshape(-1,1))
        ord_idx=np.argsort(means.cluster_centers_.flatten())
        cntrs = np.zeros_like(cluster_id)-1
        for i in np.arange(clusters):
            cntrs[cluster_id==ord_idx[i]]=i

        clusterData["nucleusUnder"] = cntrs
        
    elif chunkTime:
        
        cluster_id = means.fit_predict(data.loc[data.index<=chunkTime, channel].values.reshape(-1,1))
        ord_idx=np.argsort(means.cluster_centers_.flatten())
        cntrs = np.zeros_like(cluster_id)-1
        for i in np.arange(clusters):
            cntrs[cluster_id==ord_idx[i]]=i
        clusterData.loc[clusterData.index<=chunkTime, "nucleusUnder"] = cntrs
        cluster_id = means.fit_predict(data.loc[data.index>chunkTime, channel].values.reshape(-1,1))
        ord_idx=np.argsort(means.cluster_centers_.flatten())
        cntrs = np.zeros_like(cluster_id)-1
        for i in np.arange(clusters):
            cntrs[cluster_id==ord_idx[i]]=i
        clusterData.loc[clusterData.index>chunkTime, "nucleusUnder"] = cntrs
    elif chunkBead:
        cluster_id = means.fit_predict(data.loc[data["beadID"]<=chunkBead, channel].values.reshape(-1,1))
        ord_idx=np.argsort(means.cluster_centers_.flatten())
        cntrs = np.zeros_like(cluster_id)-1
        for i in np.arange(clusters):
            cntrs[cluster_id==ord_idx[i]]=i
        clusterData.loc[clusterData["beadID"]<=chunkBead, "nucleusUnder"] = cntrs
        
        cluster_id = means.fit_predict(data.loc[data["beadID"]>chunkBead, channel].values.reshape(-1,1))
        ord_idx=np.argsort(means.cluster_centers_.flatten())
        cntrs = np.zeros_like(cluster_id)-1
        for i in np.arange(clusters):
            cntrs[cluster_id==ord_idx[i]]=i
        clusterData.loc[clusterData["beadID"]>chunkBead, "nucleusUnder"] = cntrs
            
        

            
    return clusterData["nucleusUnder"]

In [ ]:
#Read the multichannel image, with dimensions being (Time, Z, X, Y, channel), and channels ordered as beads, actin and nucleus
images = imread(fileName)


In [ ]:
#Image Drift Correction Using Bead Plane Summation and FFT Cross-Correlation


beadsPlaneSum = np.sum(images[:,10:40,:,:,0], axis=1)

shifts = []

for time in np.arange(beadsPlaneSum[:,:,:].shape[0])[:-2]:
    corr_img = signal.fftconvolve(beadsPlaneSum[time, :,:], beadsPlaneSum[time+1, :,:][::-1,::-1], mode='same')
    shift_center = np.unravel_index(np.argmax(corr_img), corr_img.shape)
    shift = (-(corr_img.shape[0]//2-shift_center[0]), -(corr_img.shape[1]//2-shift_center[1]))
    if shift[0]!=0 or shift[1]!=0:
        images[time+1, :, :, :, :] = np.roll(images[time+1,:, :,:,:], (shift[0], shift[1]), axis=(1,2))
    shifts.append(shift)

In [ ]:
#Project channels in Z for representation and analysis

beadsSum = np.mean(images[:,:,:, :,0], axis=(0,1))
nucleusSum = np.mean(images[:,:40,:, :,2], axis=1)
actinSum = np.mean(images[:,:40,:, :,1], axis=1)

In [ ]:
#Segment beads and remove big chunks

beadsLabeled = extract_actin_cloud(beadsSum, threshFactor=1, smallObjectSize=20)
beadsLabeled = filter_objects_by_size(beadsLabeled, max_size=150)

In [ ]:
#Display identified beads

fig = plt.figure(figsize=(10,10))
beadSel = 20
beadsMasked = copy(beadsLabeled)

beadsMasked[beadsLabeled!=0] = 1

for con in find_contours(beadsMasked):
    contour = np.transpose(con)
    plt.plot(contour[1], contour[0], color="red")
beadsMasked2 = copy(beadsLabeled)
beadsMasked2 -= 1
beadsMasked2[beadsMasked2!=beadSel] = 0
try:
    contourSingle = np.transpose(find_contours(beadsMasked2)[0])
    plt.plot(contourSingle[1], contourSingle[0], color="yellow")
except:pass
plt.imshow(beadsSum, alpha=1, cmap="gray_r", vmin=100,)

plt.show()

In [ ]:
#Extract bead displacement from the movie data

data, newLabels = analyze_bead_displacement_center_boxes(images, beadsLabeled, margins=[5,20], marginBead=10, calculateNucleusMask=False,
                                            beadPlaneModes=["Centroid", "MaxIntensity"], calculateActinMask =False,
                                            nucleusTimeStart=3)

In [ ]:
#Display beads together with imaging data to verify correctness of analysis
viewer = napari.view_image(images[:, :, :, :,1], colormap="viridis", gamma=1, multiscale=False)#, blending="additive")
viewer.add_image(images[:, :, :, :,2], colormap="magma", multiscale=False, blending="additive")
viewer.add_labels(np.asarray(newLabels), multiscale=False)

In [ ]:
#Classify the actin (none, cytoplasmic or central pool) and nuclear (nucleus under or not) contributions
data["beadInPath"] =  classify_actin_contribution(data, boxMargin=20, cloud=True)
data["nucleusUnder"] =  classify_nucleus_contribution(data, boxMargin=20)

In [ ]:
#Plot bead traces with actin and nuclear intensities
beadPlane="beadPlaneKymo"
plot_profiles(data, bead=2, beadPlane=beadPlane, margin=5, nucleusIntensity="nucleusIntensity", actinIntensity="actinIntensity", savefig=True, savefigName="jumpzoom.jpg")
